In [1]:
import os, glob
import numpy as np
import pandas as pd
import pygmt
import contextily as cx

lake_export_dir = "../../Processed_Data/LakeSeries_export/"
lake_trend = pd.read_csv(
    "../../Processed_Data/Processed_Lake_correct_trend.txt",
    sep=r"\t",
    engine="python",
)
infile = "../../Processed_Data/EQ_Reservoir_Pairs_within20km.txt"
outdir = "EQ_Reservoir_Pairs_within20km_maps"
os.makedirs(outdir, exist_ok=True)

increase_green = "140/180/120"   # Olive landscape green
decrease_purple = "155/115/175"  # Muted aubergine purple

radius_km = 30
theta = np.linspace(0, 2 * np.pi, 361)

df = pd.read_csv(infile, sep=r"\t", engine="python")
df = df[df['Depth_km']<=30]

for i, row in df.iterrows():
    lat0 = float(row["LakeLat"])
    lon0 = float(row["LakeLon"])

    deg_lat = radius_km / 111.0
    deg_lon = radius_km / (111.0 * np.cos(np.deg2rad(lat0)))
    region = [
        lon0 - 1.2 * deg_lon,
        lon0 + 1.2 * deg_lon,
        lat0 - 1.2 * deg_lat,
        lat0 + 1.2 * deg_lat,
    ]
    region = [float(x) for x in region]

    dist_km = float(row["Dist_km"])
    deg_lat_1 = dist_km / 111.0
    deg_lon_1 = dist_km / (111.0 * np.cos(np.deg2rad(lat0)))
    lons = lon0 + deg_lon_1 * np.cos(theta)
    lats = lat0 + deg_lat_1 * np.sin(theta)

    lake = str(row.get("LakeName", ""))
    mag = row.get("Magnitude", "")
    t = row.get("Time", "")
    depth = row.get("Depth_km", "")
    title = f"{int(dist_km)} km to the closest lake and {int(float(depth))} km deep"
    info = f"M {mag} in {int(t)}"

    # -------- find curve file by Lake ID --------
    lake_id = str(row["LakeID"]).strip()
    if lake_id.endswith(".0"):
        lake_id = lake_id[:-2]

    trend_k = np.array(lake_trend[lake_trend["LakeID"] == int(lake_id)]["Trend"])

    pattern = os.path.join(lake_export_dir, f"ID{lake_id}_*.txt")
    matches = sorted(glob.glob(pattern))
    print(matches)
    curve_file = matches[0] if matches else None

    # -------- make figure with ONLY LEFT PANEL --------
    fig = pygmt.Figure()
    with pygmt.config(MAP_FRAME_TYPE="plain"):
        fig.tilemap(
            region=region,
            projection="M12c",
            source=cx.providers.OpenStreetMap.Mapnik,
            zoom="auto",
        )
        fig.basemap(frame=["WSen", "xaf", "yaf"])

        # lake
        fig.plot(
            x=[lon0],
            y=[lat0],
            style="c0.25c",
            fill=f"{increase_green}",
            pen=f"1p,{increase_green}",
        )

        # epicenter
        eqlon = float(row["Longitude"])
        eqlat = float(row["Latitude"])
        fig.plot(x=[eqlon], y=[eqlat], style="x0.35c", pen="2p,black")

        fig.text(
            x=lon0,
            y=lat0,
            text=lake,
            font="10p,Helvetica-Bold,black",
            justify="LB",
            offset="0.2c/0.2c",
            fill="white",
            pen=f"1p,{increase_green}",
        )

        fig.text(
            x=eqlon,
            y=eqlat,
            text=info,
            font="10p,Helvetica-Bold,black",
            justify="LB",
            offset="0.2c/0.2c",
        )

        # title at top-center
        fig.text(
            x=(region[0] + region[1]) / 2,
            y=region[3],
            text=title,
            font="12p,Helvetica-Bold,black",
            justify="TC",
            offset="0c/-1c",
        )

    safe_lake = "".join(c if c.isalnum() or c in "-_." else "_" for c in lake)[:60]
    outpng = os.path.join(outdir, f"{i:04d}_ID{lake_id}_{safe_lake}.png")
    fig.savefig(outpng, dpi=600)

print(f"Done. Wrote {len(df)} maps to: {outdir}/")

['../../Processed_Data/LakeSeries_export/ID2969_Dez.txt']
['../../Processed_Data/LakeSeries_export/ID1031_Rio_Hondo.txt']
['../../Processed_Data/LakeSeries_export/ID3539_Seyhan.txt']
['../../Processed_Data/LakeSeries_export/ID9920040_ludila_dam.txt']
['../../Processed_Data/LakeSeries_export/ID11320_Ralco.txt']
['../../Processed_Data/LakeSeries_export/ID9920051_Zipingpu.txt']
['../../Processed_Data/LakeSeries_export/ID3845_Baozhusi.txt']
['../../Processed_Data/LakeSeries_export/ID1635_Canyon.txt']
['../../Processed_Data/LakeSeries_export/ID2762_sanbanxi_dam.txt']
['../../Processed_Data/LakeSeries_export/ID1843_usoi_dam.txt']
['../../Processed_Data/LakeSeries_export/ID243_Ataturk_Dam.txt']
['../../Processed_Data/LakeSeries_export/ID703_Mangla.txt']
Done. Wrote 12 maps to: EQ_Reservoir_Pairs_within20km_maps/


In [2]:
import os
import glob
from PIL import Image
import numpy as np
import pandas as pd
import pygmt
import contextily as cx

lake_export_dir = "../../Processed_Data/LakeSeries_export/"
lake_trend = pd.read_csv(
    "../../Processed_Data/Processed_Lake_correct_trend.txt",
    sep=r"\t",
    engine="python",
)
infile = "../../Processed_Data/EQ_Reservoir_Pairs_within20km.txt"
outdir = "EQ_Reservoir_Pairs_within20km_maps"
os.makedirs(outdir, exist_ok=True)

increase_green = "140/180/120"   # Olive landscape green
decrease_purple = "155/115/175"  # Muted aubergine purple

radius_km = 30
theta = np.linspace(0, 2 * np.pi, 361)

df = pd.read_csv(infile, sep=r"\t", engine="python")
df = df[df["Depth_km"] <= 30]

# ---------------------------------------
# STEP 1: make and save individual maps
# ---------------------------------------
for i, row in df.iterrows():
    lat0 = float(row["LakeLat"])
    lon0 = float(row["LakeLon"])

    deg_lat = radius_km / 111.0
    deg_lon = radius_km / (111.0 * np.cos(np.deg2rad(lat0)))
    region = [
        lon0 - 1.2 * deg_lon,
        lon0 + 1.2 * deg_lon,
        lat0 - 1.2 * deg_lat,
        lat0 + 1.2 * deg_lat,
    ]
    region = [float(x) for x in region]

    dist_km = float(row["Dist_km"])
    deg_lat_1 = dist_km / 111.0
    deg_lon_1 = dist_km / (111.0 * np.cos(np.deg2rad(lat0)))
    lons = lon0 + deg_lon_1 * np.cos(theta)
    lats = lat0 + deg_lat_1 * np.sin(theta)

    lake = str(row.get("LakeName", "")).title()
    mag = row.get("Magnitude", "")
    t = row.get("Time", "")
    depth = row.get("Depth_km", "")
    # title = f"Distance: {int(dist_km)} km, Depth: {int(float(depth))} km"
    title = f"{int(dist_km)} km from the closest lake"
    info = f"M {mag} in {int(t)}"

    # -------- find curve file by Lake ID --------
    lake_id = str(row["LakeID"]).strip()
    if lake_id.endswith(".0"):
        lake_id = lake_id[:-2]

    trend_k = np.array(lake_trend[lake_trend["LakeID"] == int(lake_id)]["Trend"])

    pattern = os.path.join(lake_export_dir, f"ID{lake_id}_*.txt")
    matches = sorted(glob.glob(pattern))
    print(matches)
    curve_file = matches[0] if matches else None

    # -------- make figure with ONLY LEFT PANEL --------
    fig = pygmt.Figure()
    with pygmt.config(MAP_FRAME_TYPE="plain", FONT_ANNOT_PRIMARY="14p",   # tick labels
            FONT_LABEL="16p",           # axis labels
            MAP_TICK_LENGTH_PRIMARY="0.2c",):
        fig.tilemap(
            region=region,
            projection="M12c",
            source=cx.providers.OpenStreetMap.Mapnik,
            zoom="auto",
        )
        fig.basemap(frame=["WSen", "xaf", "yaf"])

        # lake
        fig.plot(
            x=[lon0],
            y=[lat0],
            style="c0.25c",
            fill='black',
            pen=f"1p,black",
        )

        # optional circle for distance
        # fig.plot(x=lons, y=lats, pen="1p,gray50,--")

        # epicenter
        eqlon = float(row["Longitude"])
        eqlat = float(row["Latitude"])
        fig.plot(x=[eqlon], y=[eqlat], style="x0.5c", pen="2p,black")
        
        print(lake)
        if lake != 'Sanbanxi Dam':
            fig.text(
                x=lon0,
                y=lat0,
                text=lake,
                font="20p,Helvetica-Bold,black",
                justify="TC",
                offset="0c/1c",
                # fill="white",
                # pen=f"1p,{increase_green}",
            )

            fig.text(
                x=eqlon,
                y=eqlat,
                text=info,
                font="20p,Helvetica-Bold,black",
                justify="TC",
                offset="0c/-0.5c",
            )
        else:
            fig.text(
                x=lon0,
                y=lat0,
                text=lake,
                font="20p,Helvetica-Bold,black",
                justify="TC",
                offset="0c/-0.5c",
                # fill="white",
                # pen=f"1p,{increase_green}",
            )

            fig.text(
                x=eqlon,
                y=eqlat,
                text=info,
                font="20p,Helvetica-Bold,black",
                justify="TC",
                offset="0c/1c",
            )

        # title at top-center
        fig.text(
            x=(region[0] + region[1]) / 2,
            y=region[3],
            text=title,
            font="20p,Helvetica-Bold,black",
            justify="TC",
            offset="0c/-1c",
        )

    safe_lake = "".join(c if c.isalnum() or c in "-_." else "_" for c in lake)[:60]
    outpng = os.path.join(outdir, f"{i:04d}_ID{lake_id}_{safe_lake}.png")
    fig.savefig(outpng, dpi=600)

print(f"Done. Wrote {len(df)} maps to: {outdir}/")

# ---------------------------------------
# STEP 2: combine all PNG maps into one PDF
# 2 columns x 6 rows = 12 maps per page
# ---------------------------------------
png_files = sorted(glob.glob(os.path.join(outdir, "*.png")))
pdf_out = os.path.join(outdir, "all_maps.pdf")

if not png_files:
    print("No PNG files found to combine into PDF.")
else:
    # US Letter landscape at 300 dpi
    page_w, page_h = 2550, 3300

    cols = 3
    rows = 4
    maps_per_page = cols * rows

    margin_x = 80
    margin_y = 60
    gap_x = 40
    gap_y = 30

    cell_w = (page_w - 2 * margin_x - (cols - 1) * gap_x) // cols
    cell_h = (page_h - 2 * margin_y - (rows - 1) * gap_y) // rows

    pages = []

    for page_start in range(0, len(png_files), maps_per_page):
        batch = png_files[page_start:page_start + maps_per_page]
        page = Image.new("RGB", (page_w, page_h), "white")

        for k, fname in enumerate(batch):
            r = k // cols
            c = k % cols

            x0 = margin_x + c * (cell_w + gap_x)
            y0 = margin_y + r * (cell_h + gap_y)

            img = Image.open(fname).convert("RGB")
            img.thumbnail((cell_w, cell_h), Image.Resampling.LANCZOS)

            paste_x = x0 + (cell_w - img.width) // 2
            paste_y = y0 + (cell_h - img.height) // 2

            page.paste(img, (paste_x, paste_y))

        pages.append(page)

    pages[0].save(
        pdf_out,
        save_all=True,
        append_images=pages[1:],
        resolution=300,
    )

    print(f"Combined {len(png_files)} maps into PDF: {pdf_out}")

['../../Processed_Data/LakeSeries_export/ID2969_Dez.txt']
Dez
['../../Processed_Data/LakeSeries_export/ID1031_Rio_Hondo.txt']
Rio Hondo
['../../Processed_Data/LakeSeries_export/ID3539_Seyhan.txt']
Seyhan
['../../Processed_Data/LakeSeries_export/ID9920040_ludila_dam.txt']
Ludila Dam
['../../Processed_Data/LakeSeries_export/ID11320_Ralco.txt']
Ralco
['../../Processed_Data/LakeSeries_export/ID9920051_Zipingpu.txt']
Zipingpu
['../../Processed_Data/LakeSeries_export/ID3845_Baozhusi.txt']
Baozhusi
['../../Processed_Data/LakeSeries_export/ID1635_Canyon.txt']
Canyon
['../../Processed_Data/LakeSeries_export/ID2762_sanbanxi_dam.txt']
Sanbanxi Dam
['../../Processed_Data/LakeSeries_export/ID1843_usoi_dam.txt']
Usoi Dam
['../../Processed_Data/LakeSeries_export/ID243_Ataturk_Dam.txt']
Ataturk Dam
['../../Processed_Data/LakeSeries_export/ID703_Mangla.txt']
Mangla
Done. Wrote 12 maps to: EQ_Reservoir_Pairs_within20km_maps/
Combined 13 maps into PDF: EQ_Reservoir_Pairs_within20km_maps/all_maps.pdf
